# Softmax Function
The softmax function converts a vector of raw scores into a probability distribution — used for multiclass classification, unlike sigmoid (binary) or ReLU (hidden layers).

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras import Sequential
from sklearn.datasets import make_blobs

In [4]:
def my_softmax(z):
    ez = np.exp(z)
    sm = ez/np.sum(ez)
    return(sm)

### The obvious organization
The model below uses softmax directly as the final layer's activation, with SparseCategoricalCrossentropy as the loss.

In [5]:
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers=centers, cluster_std=1.0, random_state=30)

In [6]:
model = Sequential([
    Dense(25,activation='relu'),
    Dense(15,activation='relu'),
    Dense(4,activation='softmax')

    
])

model.compile(
    loss= tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(0.001)     
)

model.fit(X_train,y_train,epochs =10)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 1.4864
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.8530
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6586
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4440
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1911
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1180
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0891
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0741
Epoch 9/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0652
Epoch 10/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0596


### Obvious model output
Since softmax is built into the final layer, predictions are already a probability distribution.

In [7]:
p_nonpreferred = model.predict(X_train)
print(p_nonpreferred[:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
[[1.6616279e-03 2.7398609e-03 9.5578569e-01 3.9812777e-02]
 [9.8788851e-01 1.5492011e-03 1.0155333e-02 4.0687001e-04]]
largest value 0.9999981 smallest value 2.1074854e-10


### Preferred organization
More numerically stable results are obtained by using a linear output layer and letting the loss function handle softmax internally via from_logits=True.

In [8]:
preferred_model = Sequential([
    Dense(25, activation='relu'),
    Dense(15, activation='relu'),
    Dense(4, activation='linear')
])
preferred_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(0.001),
)
preferred_model.fit(X_train, y_train, epochs=10)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.0732
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3863
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1799
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1056
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0776
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0637
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0552
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0495
Epoch 9/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0450
Epoch 10/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0416


### Output handling
The preferred model's raw outputs are not probabilities — apply softmax separately to convert them.

In [9]:
p_preferred = preferred_model.predict(X_train)
print(f"two example output vectors:\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
two example output vectors:
 [[-2.6624877  -2.2628074   3.2458787  -0.64925694]
 [ 4.6567483  -0.41072547 -3.8952796  -6.1894    ]]
largest value 11.713107 smallest value -9.969288


In [10]:
#apply softmax manually, after raw output

sm_preferred = tf.nn.softmax(p_preferred).numpy()  # .numpy() converts tensorflow array to simple array
print(f"two example output vectors:\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))

two example output vectors:
 [[2.6449217e-03 3.9444976e-03 9.7360682e-01 1.9803764e-02]
 [9.9353129e-01 6.2575680e-03 1.9190359e-04 1.9353483e-05]]
largest value 0.99999845 smallest value 5.497405e-10


### Selecting the predicted category
To pick the most likely class, softmax isn't even required — just find the index of the largest raw output using np.argmax().

In [11]:
for i in range(5):
    print(f"{p_preferred[i]}, category: {np.argmax(p_preferred[i])}")

[-2.6624877  -2.2628074   3.2458787  -0.64925694], category: 2
[ 4.6567483  -0.41072547 -3.8952796  -6.1894    ], category: 0
[ 3.4283607   0.07820506 -2.9171677  -4.896573  ], category: 0
[-1.6933569   4.6167445   0.06669049 -2.121249  ], category: 1
[-1.8249288 -2.622629   3.87208   -4.1635966], category: 2


### Summary
This lab implemented the softmax function by hand, then built two versions of a multiclass classifier in TensorFlow: the obvious method (softmax in the final layer) and the preferred method (linear output + from_logits=True), which is more numerically stable. np.argmax() was used to pick the predicted class without needing to compute probabilities explicitly.